FIgure 3.1

In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define paths for the ZIP file and local extraction directory
zip_path = '/content/drive/MyDrive/Brain_Tumor_Dataset.zip'
extract_path = '/content/local_tumor_data'

# 3. Unzip directly to Colab local storage to significantly boost I/O speed during training
if not os.path.exists(extract_path):
    print("Extracting dataset to local environment...")
    !unzip -q "{zip_path}" -d "{extract_path}"
    print("Extraction complete. Dataset is ready.")
else:
    print("Dataset already exists in local environment.")

Mounted at /content/drive
Extracting dataset to local environment...
Extraction complete. Dataset is ready.


Figure 3.2ab

In [ ]:

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define dataset directories pointing to the fast local storage
train_dir = '/content/local_tumor_data/Brain_Tumor_Dataset/Training'
test_dir = '/content/local_tumor_data/Brain_Tumor_Dataset/Testing'

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Implement Data Augmentation with a validation split
train_datagen = ImageDataGenerator(
    rescale=1./255,           # Normalize pixel values to [0,1] range
    rotation_range=15,          # Apply minor rotation for augmentation
    width_shift_range=0.1,        # Horizontal spatial shift
    height_shift_range=0.1,       # Vertical spatial shift
    horizontal_flip=True,        # Allow horizontal flipping
    validation_split=0.2         # Reserve 20% of data for validation
)

# Testing set requires normalization only
test_datagen = ImageDataGenerator(rescale=1./255)

print("\n--- Initializing Training Generator ---")
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

print("\n--- Initializing Validation Generator ---")
val_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)


--- Initializing Training Generator ---
Found 4480 images belonging to 4 classes.

--- Initializing Validation Generator ---
Found 1120 images belonging to 4 classes.


Figure 3.3 & Figure 3.4

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout

# 1. Define Input Layer
input_tensor = Input(shape=(224, 224, 3))

# 2. Instantiate pre-trained ResNet50 base
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_tensor=input_tensor
)

# 3. Freeze pre-trained convolutional layers initially
base_model.trainable = False

# 4. Construct custom classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)           # Dimensionality reduction of feature maps
x = Dense(128, activation='relu')(x)        # Fully connected layer with 128 units
x = Dropout(0.4)(x)                 # Regularization (40% dropout) to prevent overfitting
predictions = Dense(4, activation='softmax')(x)  # Output layer for 4 distinct tumor classes

# 5. Compile the final functional model
model = Model(inputs=base_model.input, outputs=predictions)
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
# Generate architecture summary for reporting
model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 23,850,500 (90.98 MB)

 Trainable params: 262,788 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

### Member 4 - Training


In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Callbacks for early stopping and model checkpointing
early_stopping = EarlyStopping(
    monitor='val_loss',     # Monitor validation loss
    patience=5,             # Number of epochs with no improvement after which training will be stopped
    restore_best_weights=True # Restore model weights from the epoch with the best value of the monitored quantity
)

model_checkpoint = ModelCheckpoint(
    'best_brain_tumor_model.keras', # Path to save the best model
    monitor='val_accuracy',      # Monitor validation accuracy
    save_best_only=True,         # Only save the best model
    mode='max',                  # Save model when validation accuracy is maximized
    verbose=1
)

In [ ]:
;history = model.fit(
    train_generator,
    epochs=20, # Start with a reasonable number of epochs, EarlyStopping will handle the rest
    validation_data=val_generator,
    callbacks=[early_stopping, model_checkpoint]
)

Epoch 1/20
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 455ms/step - accuracy: 0.3531 - loss: 1.3585
Epoch 1: val_accuracy improved from None to 0.50179, saving model to best_brain_tumor_model.keras

Epoch 1: finished saving model to best_brain_tumor_model.keras
140/140 ━━━━━━━━━━━━━━━━━━━━ 100s 608ms/step - accuracy: 0.4065 - loss: 1.2312 - val_accuracy: 0.5018 - val_loss: 1.1131
Epoch 2/20
 31/140 ━━━━━━━━━━━━━━━━━━━━ 45s 418ms/step - accuracy: 0.4413 - loss: 1.1709

In [ ]:
import matplotlib.pyplot as plt

# Plotting training and validation accuracy
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Plotting training and validation loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Save the final trained model (or the best one saved by ModelCheckpoint)
model.save('final_brain_tumor_model.keras')
print("Model saved as final_brain_tumor_model.keras")

Member 5 — Evaluation and Metrics

In [ ]:
import os
print("File present:", os.path.exists('final_brain_tumor_model.keras'))

In [ ]:
!gdown 1rDm1MxdHmdvSjDYNCaXuVAiT2MAlRNxO -O final_brain_tumor_model.keras

Figure 7.1 - Loading the trained model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    roc_curve,
    auc,
)

model = tf.keras.models.load_model('final_brain_tumor_model.keras')
print("Model loaded. Input shape:", model.input_shape)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import glob
for f in glob.glob('/content/drive/MyDrive/**/*.zip', recursive=True):
    print(f)

In [ ]:
import os, zipfile

zip_path = '/content/drive/MyDrive/archive (3).zip'
extract_path = '/content/tumor_data'

# Unzip only if not already done
if not os.path.exists(extract_path):
    print("Extracting...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_path)
    print("Done.")
else:
    print("Already extracted.")

# Find the Testing folder wherever it landed inside the zip
test_dir = None
for root, dirs, files in os.walk(extract_path):
    if os.path.basename(root).lower() in ('testing', 'test'):
        test_dir = root
        break

print("test_dir =", test_dir)

Figure 7.2 - Preparing the test set

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

class_indices = test_generator.class_indices
class_names = [name for name, idx in sorted(class_indices.items(), key=lambda kv: kv[1])]
print("Class order used by the model:", class_names)

Figure 7.3 - Running predictions on the test set

In [ ]:
test_generator.reset()

y_prob = model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_prob, axis=1)
y_true = test_generator.classes

print("Predictions:", y_prob.shape, "| True labels:", y_true.shape)

Figure 7.4 - Accuracy and per-class metrics

In [ ]:
overall_acc = accuracy_score(y_true, y_pred)
print(f"Overall Test Accuracy: {overall_acc:.4f}\n")

report = classification_report(
    y_true, y_pred,
    target_names=class_names,
    digits=4
)
print(report)

Figure 7.5 - Confusion matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title('Confusion Matrix (counts)')
axes[0].set_xlabel('Predicted label')
axes[0].set_ylabel('True label')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title('Confusion Matrix (row-normalised)')
axes[1].set_xlabel('Predicted label')
axes[1].set_ylabel('True label')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

Figure 7.6 - Per-class accuracy

In [ ]:
print("Per-class accuracy (recall per class):")
for i, name in enumerate(class_names):
    print(f"  {name:12s}: {cm_norm[i, i]:.4f}  ({cm[i, i]} / {cm[i].sum()} correct)")

Figure 7.7 - ROC curves and AUC

In [ ]:
y_true_onehot = tf.keras.utils.to_categorical(y_true, num_classes=len(class_names))

plt.figure(figsize=(8, 6))
for i, name in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_true_onehot[:, i], y_prob[:, i])
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc(fpr, tpr):.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('One-vs-Rest ROC Curves per Tumour Class')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()